# Tarea 1: prediccion de resultados del futbol uruguayo

Notebook autocontenido para limpiar los datos, crear atributos causales, seleccionar hiperparametros con validacion temporal y evaluar en 2024-2025. Ejecutar todas las celdas desde la raiz del repositorio.

## 1. Configuracion reproducible

La semilla utilizada por los modelos estocasticos es 42. Las versiones exigidas se declaran en `pyproject.toml`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

from aa_futbol.data import DEFAULT_INPUT, prepare_dataset
from aa_futbol.experiments import LABELS, run_experiments
from aa_futbol.features import build_causal_match_features
from aa_futbol.model_selection import DateBlockedTimeSeriesSplit

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
RESULTS = ROOT / 'results'
pd.set_option('display.max_columns', 30)

## 2. Limpieza desde la fuente original

No se lee ningun CSV procesado manualmente. `winner` se deriva exclusivamente de `gh` y `ga`; esas columnas no seran atributos predictivos.

In [ ]:
matches, cleaning_report = prepare_dataset()
display(pd.Series(cleaning_report, name='valor'))
display(matches.head())

In [ ]:
class_distribution = (
    matches['winner'].value_counts().reindex(LABELS).rename('cantidad').to_frame()
)
class_distribution['porcentaje'] = 100 * class_distribution['cantidad'] / len(matches)
display(class_distribution)
matches.groupby(matches['date'].dt.year)['winner'].count().plot(figsize=(10, 3), title='Partidos por anio')
plt.ylabel('Partidos')
plt.show()

## 3. Atributos historicos causales

Las tasas, puntos y diferencias de gol se calculan con fechas estrictamente anteriores. Todos los partidos del mismo dia se featurizan antes de actualizar historiales.

In [ ]:
featured = build_causal_match_features(matches)
train = featured[featured['year'] <= 2023].reset_index(drop=True)
test = featured[featured['year'].between(2024, 2025)].reset_index(drop=True)
display(featured.head())
print(f'Train: {len(train):,} partidos ({train.date.min().date()} a {train.date.max().date()})')
print(f'Test:  {len(test):,} partidos ({test.date.min().date()} a {test.date.max().date()})')

## 4. Verificacion de los folds temporales

In [ ]:
splitter = DateBlockedTimeSeriesSplit(n_splits=5)
fold_rows = []
X_train = train.drop(columns='winner')
for fold, (fit_index, validation_index) in enumerate(splitter.split(X_train), start=1):
    fold_rows.append({
        'fold': fold,
        'train_rows': len(fit_index),
        'train_end': X_train.loc[fit_index, 'date'].max().date(),
        'validation_rows': len(validation_index),
        'validation_start': X_train.loc[validation_index, 'date'].min().date(),
        'validation_end': X_train.loc[validation_index, 'date'].max().date(),
    })
display(pd.DataFrame(fold_rows))

## 5. Seleccion de hiperparametros y evaluacion final

Durante desarrollo se puede usar `profile='quick'`. Para las tablas finales cambiar a `profile='full'`, ejecutar nuevamente y conservar las salidas revisadas.

In [ ]:
PROFILE = 'quick'  # Cambiar a 'full' para la corrida final.
summary, details, models, featured = run_experiments(profile=PROFILE)
display(summary)

In [ ]:
for model_name in summary['model']:
    path = RESULTS / f'confusion_{model_name}.png'
    print(model_name)
    display(Image(filename=path))

## 6. Clasificacion de instancias y analisis cualitativo

La tabla permite buscar partidos donde los modelos discrepan, empates no detectados y patrones de error del mejor modelo.

In [ ]:
predictions = pd.read_csv(RESULTS / 'test_predictions.csv', parse_dates=['date'])
prediction_columns = [column for column in predictions if column.startswith('prediction_')]
predictions['models_correct'] = predictions[prediction_columns].eq(predictions['winner'], axis=0).sum(axis=1)
display(predictions.sample(10, random_state=42))
display(predictions.sort_values('models_correct').head(20))

## 7. Pendientes para el informe

- Ejecutar el perfil completo y registrar versiones.
- Explicar el efecto de `m` y `min_info_gain` usando las curvas.
- Comparar todas las metricas, no solo accuracy.
- Elegir ejemplos concretos para el analisis cualitativo.
- Discutir limitaciones de datos, atributos, baseline y protocolo online.
- Adaptar la declaracion de IA del archivo `AI_USAGE.md`.